# Dive into RLHF
> This lab manual translates and integrates online resources [blog](https://newfacade.github.io/notes-on-reinforcement-learning/17-ppo-trl.html) & [trl examples](https://github.com/huggingface/trl/blob/main/examples/notebooks/gpt2-sentiment.ipynb)

Reproduction experiment configuration: a single NVIDIA A800-SXM4-80GB uses 10097MiB, and training takes 35min19s.

## How PPO Works
1. Rollout: the language model generates responses based on queries.
2. Evaluation: queries and responses are evaluated using functions, models, human feedback, or some combination of them. This process should produce a **scalar value** for each query/response pair.
3. Optimization: in the optimization step, query/response pairs are used to calculate the log probabilities of the tokens in the sequence. This is done with the trained model and the reference model. The KL divergence between the two outputs is used as an additional reward signal to ensure that the generated responses do not drift too far from the reference language model. PPO is then used to train the active language model.
<div style="text-align: center">
<img src='figs/trl1.png' width='600'>
<p style="text-align: center;"> <b>Figure:</b> PPO workflow </p>
</div>

# Fine-Tuning GPT-2 to Generate Positive Reviews  
> Optimize GPT-2 to generate positive IMDB movie reviews by using a BERT sentiment classifier as the reward function.

<div style="text-align: center">
<img src='figs/gpt2_bert_training.png' width='600'>
<p style="text-align: center;"> <b>Figure:</b> Experimental setup for fine-tuning GPT-2</p>
</div>

We fine-tune GPT-2 to generate positive movie reviews based on the IMDB dataset. The model receives the beginning of a real review and needs to generate a positive continuation. To reward positive continuations, we use a BERT classifier to analyze the sentiment of the generated sentences and use the classifier's output as the reward signal for PPO training.

## Experimental Setup

### Download Models and Data
Dataset
```bash
export HF_ENDPOINT=https://hf-mirror.com; huggingface-cli download --resume-download stanfordnlp/imdb --local-dir dataset/imdb --repo-type dataset
```
Reference model
```bash
export HF_ENDPOINT=https://hf-mirror.com; huggingface-cli download --resume-download lvwerra/gpt2-imdb --local-dir model/gpt2-imdb
```
Reward model
```bash
export HF_ENDPOINT=https://hf-mirror.com; huggingface-cli download --resume-download lvwerra/distilbert-imdb --local-dir model/distilbert-imdb
```

### Import Dependencies

In [ ]:
# %pip install -r requirements.txt
# import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [2]:
import torch
from tqdm import tqdm
import pandas as pd

tqdm.pandas()

from transformers import pipeline, AutoTokenizer
from datasets import load_dataset

from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler

### Configuration

In [3]:
config = PPOConfig(
    model_name="model/gpt2-imdb",
    learning_rate=1.41e-5,
    log_with="wandb",
)

sent_kwargs = {"top_k": None, "function_to_apply": "none", "batch_size": 16}

In [ ]:
import wandb

wandb.init()

You can see that we loaded a GPT-2 model named `gpt2_imdb`. This model was additionally fine-tuned for 1 epoch on the IMDB dataset using Hugging Face's [script](https://github.com/huggingface/transformers/blob/main/examples/legacy/run_language_modeling.py) (with no special settings). The remaining parameters are mainly taken from the original paper "[Fine-Tuning Language Models from Human Preferences](https://huggingface.co/papers/1909.08593)". Both this model and the BERT model are available in the Hugging Face model hub at [this link](https://huggingface.co/models).

## Loading Data and Models

### Load the IMDB Dataset  
The IMDB dataset contains 50,000 movie reviews annotated with "positive"/"negative" sentiment feedback. We load the IMDB dataset into a DataFrame and filter for reviews with at least 200 characters. Then, we tokenize each text and use `LengthSampler` to randomly truncate it to a specified length.

In [5]:
def build_dataset(
    config,
    dataset_name="dataset/imdb",
    input_min_text_length=2,
    input_max_text_length=8,
):
    """
    Build dataset for training. This builds the dataset from `load_dataset`, one should
    customize this function to train the model on its own dataset.

    Args:
        dataset_name (`str`):
            The name of the dataset to be loaded.

    Returns:
        dataloader (`torch.utils.data.DataLoader`):
            The dataloader for the dataset.
    """
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    tokenizer.pad_token = tokenizer.eos_token
    # load imdb with datasets
    ds = load_dataset(dataset_name, split="train")
    ds = ds.rename_columns({"text": "review"})
    ds = ds.filter(lambda x: len(x["review"]) > 200, batched=False)

    input_size = LengthSampler(input_min_text_length, input_max_text_length)

    def tokenize(sample):
        sample["input_ids"] = tokenizer.encode(sample["review"])[: input_size()]
        sample["query"] = tokenizer.decode(sample["input_ids"])
        return sample

    ds = ds.map(tokenize, batched=False)
    ds.set_format(type="torch")
    return ds

In [6]:
dataset = build_dataset(config)


def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

### Load the Pretrained GPT-2 Language Model
We load the GPT-2 model with a value head and the tokenizer. We load the model twice; the first model is used for optimization, while the second model serves as a reference to compute the KL divergence from the initial point. This acts as an additional reward signal in PPO training to ensure that the optimized model does not drift too far from the original language model.

In [7]:
model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

tokenizer.pad_token = tokenizer.eos_token

### Initialize PPOTrainer  
`PPOTrainer` handles subsequent device allocation and optimization:

In [8]:
ppo_trainer = PPOTrainer(
    config, model, ref_model, tokenizer, dataset=dataset, data_collator=collator
)

### Load the BERT Classifier  
We load a BERT classifier fine-tuned on the IMDB dataset.

In [9]:
device = ppo_trainer.accelerator.device
if ppo_trainer.accelerator.num_processes == 1:
    device = 0 if torch.cuda.is_available() else "cpu"  # to avoid a `pipeline` bug
sentiment_pipe = pipeline(
    "sentiment-analysis", model="model/distilbert-imdb", device=device
)

Device set to use cuda:0


The model outputs logits for the negative and positive classes. We will use the positive-class logits as the reward signal for the language model.

In [10]:
text = "this movie was really bad!!"
sentiment_pipe(text, **sent_kwargs)

[{'label': 'NEGATIVE', 'score': 2.3350484371185303},
 {'label': 'POSITIVE', 'score': -2.726576089859009}]

In [11]:
text = "this movie was really good!!"
sentiment_pipe(text, **sent_kwargs)

[{'label': 'POSITIVE', 'score': 2.557040214538574},
 {'label': 'NEGATIVE', 'score': -2.294790267944336}]

### Generation Settings  
For response generation, we only use sampling, ensuring that top-k and nucleus sampling are disabled while setting a minimum length.

In [12]:
gen_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}

## Optimizing the Model

### Training Loop

The training loop includes the following main steps:
1. Obtain query responses from the policy network (GPT-2)  
2. Obtain the sentiment of the query/response from BERT  
3. Optimize the policy with PPO using (query, response, reward) triples  
<!-- **Training Time**  
Under the settings specified above, this step takes **about 2 hours** on a V100 GPU. -->

In [13]:
output_min_length = 4
output_max_length = 16
output_length_sampler = LengthSampler(output_min_length, output_max_length)


generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}


for epoch, batch in enumerate(tqdm(ppo_trainer.dataloader)):
    query_tensors = batch["input_ids"]

    #### Get response from gpt2
    response_tensors = []
    for query in query_tensors:
        gen_len = output_length_sampler()
        generation_kwargs["max_new_tokens"] = gen_len
        query_response = ppo_trainer.generate(query, **generation_kwargs).squeeze()
        response_len = len(query_response) - len(query)
        response_tensors.append(query_response[-response_len:])
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    #### Compute sentiment score
    texts = [q + r for q, r in zip(batch["query"], batch["response"])]
    pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
    positive_scores = [
        item["score"]
        for output in pipe_outputs
        for item in output
        if item["label"] == "POSITIVE"
    ]
    rewards = [torch.tensor(score) for score in positive_scores]

    #### Run PPO step
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
    ppo_trainer.log_stats(stats, batch, rewards)

100%|██████████| 194/194 [35:19<00:00, 10.92s/it]


### Training Progress  
If you are using Weights & Biases to track training progress, you should see a curve similar to the one below. View the interactive example report on wandb.ai: [link](https://wandb.ai/huggingface/trl/runs/w9l3110g).  
<div style="text-align: center">
<img src='figs/gpt2_tuning_progress.png' width='800'>
<p style="text-align: center;"> <b>Figure:</b> Evolution of mean reward during training </p>
</div>  
It can be observed that after several optimization steps, the model begins to generate more positive outputs.  

<!-- > Note: if you examine the KL divergence, you may find that the model has not yet converged to the target KL divergence at this point. Reaching the target requires longer training time or a higher initial coefficient. -->

## Model Inspection  
Let's inspect some examples from the IMDB dataset. We can use `ref_model` to compare the optimized model `model` with the model before optimization.

In [14]:
#### get a batch from the dataset
bs = 16
game_data = dict()
dataset.set_format("pandas")
df_batch = dataset[:].sample(bs)
game_data["query"] = df_batch["query"].tolist()
query_tensors = df_batch["input_ids"].tolist()

response_tensors_ref, response_tensors = [], []

#### get response from gpt2 and gpt2_ref
for i in range(bs):
    query = torch.tensor(query_tensors[i]).to(device)

    gen_len = output_length_sampler()
    query_response = ref_model.generate(
        query.unsqueeze(0), max_new_tokens=gen_len, **gen_kwargs
    ).squeeze()
    response_len = len(query_response) - len(query)
    response_tensors_ref.append(query_response[-response_len:])

    query_response = model.generate(
        query.unsqueeze(0), max_new_tokens=gen_len, **gen_kwargs
    ).squeeze()
    response_len = len(query_response) - len(query)
    response_tensors.append(query_response[-response_len:])

#### decode responses
game_data["response (before)"] = [
    tokenizer.decode(response_tensors_ref[i]) for i in range(bs)
]
game_data["response (after)"] = [
    tokenizer.decode(response_tensors[i]) for i in range(bs)
]

#### sentiment analysis of query/response pairs before/after
texts = [q + r for q, r in zip(game_data["query"], game_data["response (before)"])]
pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
positive_scores = [
    item["score"]
    for output in pipe_outputs
    for item in output
    if item["label"] == "POSITIVE"
]
game_data["rewards (before)"] = positive_scores

texts = [q + r for q, r in zip(game_data["query"], game_data["response (after)"])]
pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
positive_scores = [
    item["score"]
    for output in pipe_outputs
    for item in output
    if item["label"] == "POSITIVE"
]
game_data["rewards (after)"] = positive_scores

# store results in a dataframe
df_results = pd.DataFrame(game_data)
df_results

,query,response (before),response (after),rewards (before),rewards (after)
0,Well I guess I know,that Cantor may be an,..but I loved it,0.230196,2.281557
1,"This is an excellent,",direct-to-video film with typical,enjoyable movie.<|endoftext|>,2.846593,2.840860
2,"Now, I",'ve never had the chance with James,loved the growing episode - and the,0.656194,2.525894
3,We tend,not to see Arthur,to like this very,-0.280880,2.183822
4,"The proverb ""Never judge a book","by the cover"" has caught on. After glancing t...","with high compliments, but it is recommended ...",0.274649,2.065951
5,I've never understood,"why so many artsmen,",this film but it's delightful,0.835574,2.782384
6,Hugh (Ed Harris) is,"an acclaimed ""hero"" and his fian",a wonderful actor who is a good adaptation,1.580167,2.602940
7,This particular Joe McDoakes,' episode brought all the wrong bits and,movie is really a great movie. It,0.870956,2.795245
8,Sisters In,"Vrooms 8.23, I signed up for all of the","The Universe 1: Sunny is cute, and has a cute...",1.175259,2.062330
9,I was very fond of this,"film, it was obviously a bad idea when first ...","show, and know that I have seen it several times",1.058164,2.511273


By observing the mean/median rewards of the generated sequences, we found a significant difference.

In [15]:
print("mean:")
display(df_results[["rewards (before)", "rewards (after)"]].mean())
print()
print("median:")
display(df_results[["rewards (before)", "rewards (after)"]].median())

mean:


rewards (before)    0.591666
rewards (after)     2.243934
dtype: float64


median:


rewards (before)    0.646912
rewards (after)     2.492161
dtype: float64

## Saving the Model  
Finally, we save the model for later use.

In [16]:
model.save_pretrained("model/gpt2-imdb-pos-v2")
tokenizer.save_pretrained("model/gpt2-imdb-pos-v2")

('model/gpt2-imdb-pos-v2/tokenizer_config.json',
 'model/gpt2-imdb-pos-v2/special_tokens_map.json',
 'model/gpt2-imdb-pos-v2/vocab.json',
 'model/gpt2-imdb-pos-v2/merges.txt',
 'model/gpt2-imdb-pos-v2/added_tokens.json',
 'model/gpt2-imdb-pos-v2/tokenizer.json')